# Step 4 — Train YOLOv8 on the VLM-generated labels

We fine-tune `yolov8n.pt` on the 150-image training set that the VLM labeled in notebook 02. The point isn't state-of-the-art accuracy — it's to prove that VLM-generated annotations are good enough to train a real, small, fast detector that we can then deploy on the edge or on RHOAI.

**Input:** `datasets/vlm-annotated/dataset.yaml` from notebook 03.  
**Output:** `runs/detect/ppe-vlm/weights/best.pt` — consumed by [`05-register-and-deploy.ipynb`](05-register-and-deploy.ipynb).

In [ ]:
%%capture
!pip install ultralytics torch torchvision

In [ ]:
from pathlib import Path

import torch
from ultralytics import YOLO

DATA_YAML = Path("datasets/vlm-annotated/dataset.yaml").resolve()
RUN_NAME  = "ppe-vlm"
PROJECT   = "runs/detect"

assert DATA_YAML.is_file(), f"Missing {DATA_YAML} — run notebook 03 first."
print("CUDA available:", torch.cuda.is_available())

## Train

20 epochs on ~150 images with `yolov8n.pt` initialization is a ~3–5 minute run on a single GPU, or ~15 minutes on CPU. Adjust `epochs` down to 5 if you're time-boxed in a live demo.

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=20,
    imgsz=640,
    batch=16,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    patience=10,
)

## Quick evaluation

In [ ]:
best_pt = Path(PROJECT) / RUN_NAME / "weights" / "best.pt"
print(f"Best weights: {best_pt}")

trained = YOLO(str(best_pt))
metrics = trained.val(data=str(DATA_YAML), imgsz=640)
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

## Sanity predict on one of the samples/ images

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample_img = Path("samples/ppe-01.jpg")
pred = trained.predict(str(sample_img), imgsz=640, conf=0.25, save=False)[0]
annotated = Image.fromarray(pred.plot()[:, :, ::-1])
plt.figure(figsize=(10, 8))
plt.imshow(annotated)
plt.axis("off")
plt.title(f"YOLOv8n fine-tuned on VLM labels — {sample_img.name}")
plt.show()

---

**Next:** [05-register-and-deploy.ipynb](05-register-and-deploy.ipynb) — push `best.pt` to S3, register it in the RHOAI Model Registry, and deploy it as a KServe `InferenceService`.